# Traffic Fatality Prediction - Complete Pipeline

This notebook integrates the entire pipeline for traffic fatality prediction including:
- Data loading and sampling (10% of original dataset)
- Seed finding for reproducibility
- Train/test split
- Data preprocessing and imputation
- Grid search for mixed sampling parameters
- Sampling techniques application
- Visualization and analysis
- Model evaluation

## 1. Setup & Installation

Install R packages for MICE and ROSE imputation

In [ ]:
!Rscript -e "install.packages(c('mice','ROSE'), repos='https://cloud.r-project.org')"

In [ ]:
import rpy2
%load_ext rpy2.ipython

In [ ]:
%%R
library(mice)
library(ROSE)

In [ ]:
import os
import warnings
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.stats import chisquare, ks_2samp, chi2_contingency

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 200
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.figsize"] = (8, 5)

# Kaggle paths
DATA_DIR = "/kaggle/input"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("Environment ready.")

## 2. Data Loading & Sampling

In [ ]:
# Load all CSV files from Kaggle input
files = sorted(glob.glob(f"{DATA_DIR}/**/*.csv*", recursive=True))

df_list = []
for file in files:
    df = pd.read_csv(file, dtype=str, low_memory=False)
    df_list.append(df)
    print(f"{os.path.basename(file):20} {len(df):,} rows")

df_full = pd.concat(df_list, ignore_index=True)
print(f"\nTotal: {len(df_full):,} rows")

In [ ]:
# Sample 10% of original dataset
SAMPLE_RATIO = 0.1
RANDOM_STATE_SAMPLE = 42

n_samples = int(len(df_full) * SAMPLE_RATIO)
print(f"Sampling {n_samples:,} rows ({SAMPLE_RATIO*100:.1f}% of original)")

df_sampled = df_full.sample(n=n_samples, random_state=RANDOM_STATE_SAMPLE).reset_index(drop=True)
print(f"Sampled dataset: {len(df_sampled):,} rows")

## 3. Seed Finding

In [ ]:
NA_CODES = {"U", "UU", "UUUU", "X", "XX", "XXXX", "N", "NN", "NNNN", "Q", "QQ"}

def clean_numeric(series):
    return pd.to_numeric(series.replace(NA_CODES, np.nan), errors="coerce")

# Create target variable
num_cols_all = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_SEV', 'C_VEHS',
                'C_CONF', 'C_RCFG', 'C_WTHR', 'C_RSUR', 'C_RALN', 'C_TRAF',
                'V_ID', 'V_TYPE', 'V_YEAR', 'P_ID', 'P_AGE', 'P_PSN',
                'P_SEX', 'P_SAFE', 'P_ISEV', 'P_USER']

df_sampled['P_SEX'] = df_sampled['P_SEX'].replace({'M': 1, 'F': 0})
for col in num_cols_all:
    if col in df_sampled.columns:
        df_sampled[col] = clean_numeric(df_sampled[col])

df_sampled['Fatality'] = (df_sampled['P_ISEV'] == 3).astype(int)
print(f"Fatality rate: {df_sampled['Fatality'].mean()*100:.3f}%")

In [ ]:
# Seed search configuration
N_SEEDS = 3
MAX_ATTEMPTS = 50

cat_cols = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_SEV', 'C_VEHS',
            'C_CONF', 'C_RCFG', 'C_WTHR', 'C_RSUR', 'C_RALN', 'C_TRAF',
            'V_TYPE', 'P_SEX', 'P_PSN', 'P_SAFE', 'P_USER', 'Fatality']
num_cols = ['P_AGE', 'V_YEAR']

def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

def evaluate_sample(pop_series, samp_series, col, col_type='cat'):
    if col_type == 'cat':
        pop_dist = pop_series.value_counts(normalize=True).sort_index()
        samp_dist = samp_series.value_counts(normalize=True).sort_index()
        common = pop_dist.index.intersection(samp_dist.index)
        if len(common) < 2:
            return None
        observed = (samp_dist[common] * len(samp_series)).fillna(0).values
        expected_prop = (pop_dist[common] / pop_dist[common].sum()).values
        expected = expected_prop * len(samp_series)
        _, p_chi = chisquare(observed, expected)
        ct = pd.crosstab(pop_series, samp_series, dropna=False)
        cv = cramers_v(ct.values) if ct.shape == (2, 2) else 0
        return {'column': col, 'type': 'categorical', 'chi2_p': p_chi, 'cramers_v': cv}
    else:
        pop_vals = pop_series.dropna()
        samp_vals = samp_series.dropna()
        if len(pop_vals) < 5 or len(samp_vals) < 5:
            return None
        ks_stat, ks_p = ks_2samp(pop_vals, samp_vals)
        return {'column': col, 'type': 'numeric', 'ks_stat': ks_stat, 'ks_p': ks_p}

In [ ]:
# Seed search
print("=" * 60)
print("SEED SEARCH")
print(f"Target: {N_SEEDS} seeds")
print("=" * 60)

df_ref = df_sampled.dropna(subset=['Fatality', 'P_ISEV']).copy()

VALID_SEEDS = []
tried = 0

while len(VALID_SEEDS) < N_SEEDS and tried < MAX_ATTEMPTS:
    seed = tried + 1
    tried += 1
    
    df_s = df_ref.sample(n=min(50000, len(df_ref)), random_state=seed)
    
    all_p = []
    for col in cat_cols:
        result = evaluate_sample(df_ref[col], df_s[col], col, 'cat')
        if result:
            all_p.append(result['chi2_p'])
    for col in num_cols:
        result = evaluate_sample(df_ref[col], df_s[col], col, 'num')
        if result:
            all_p.append(result['ks_p'])
    
    n_tests = len(all_p)
    alpha = 0.05 / max(n_tests, 1)
    passed = all(p > alpha for p in all_p) if all_p else False
    
    if passed:
        VALID_SEEDS.append(seed)
        status = "PASS"
    else:
        status = "FAIL"
    
    print(f"  Seed {seed:3d}: {status}  [{len(VALID_SEEDS)}/{N_SEEDS}]")

print(f"\nFound {len(VALID_SEEDS)} valid seeds: {VALID_SEEDS}")
if len(VALID_SEEDS) < N_SEEDS:
    print(f"Using all available seeds")
SEEDS = VALID_SEEDS[:N_SEEDS]

## 4. Train/Test Split

In [ ]:
TRAIN_RATIO = 0.7
RANDOM_STATE_SPLIT = 42

print("=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)

df_train, df_test = train_test_split(
    df_sampled,
    train_size=TRAIN_RATIO,
    stratify=df_sampled['Fatality'],
    random_state=RANDOM_STATE_SPLIT
)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Train set: {len(df_train):,} samples (fatal={df_train['Fatality'].sum():,}, rate={df_train['Fatality'].mean()*100:.3f}%)")
print(f"Test set: {len(df_test):,} samples (fatal={df_test['Fatality'].sum():,}, rate={df_test['Fatality'].mean()*100:.3f}%)")

# Save train/test
train_path = os.path.join(OUT_DIR, "train.csv")
test_path = os.path.join(OUT_DIR, "test.csv")
df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)
print(f"\nSaved train to: {train_path}")
print(f"Saved test to: {test_path}")

## 5. Preprocessing

In [ ]:
def story_imputation(df):
    rain_codes = [3, 4, 5]
    mask_rain = df['C_WTHR'].isin(rain_codes) & (df['C_RSUR'].isnull() | (df['C_RSUR'] == 9))
    df.loc[mask_rain, 'C_RSUR'] = 2
    
    clear_codes = [1, 2]
    mask_clear = df['C_WTHR'].isin(clear_codes) & (df['C_RSUR'].isnull() | (df['C_RSUR'] == 9))
    df.loc[mask_clear, 'C_RSUR'] = 1
    
    mask_road = (df['C_RCFG'] == 1) & (df['C_RALN'].isnull() | (df['C_RALN'] == 9))
    df.loc[mask_road, 'C_RALN'] = 1
    
    mask_not_road = (df['C_RCFG'] != 1) & (df['C_RALN'].isnull() | (df['C_RALN'] == 9))
    df.loc[mask_not_road, 'C_RALN'] = 2
    
    print("Story imputation done")
    return df

In [ ]:
def mice_imputation(df, n_iter=5):
    import rpy2.robjects as ro
    import tempfile
    
    mice_cols = ['C_WTHR', 'C_RSUR', 'C_CONF', 'C_RCFG', 'C_RALN',
                 'C_TRAF', 'V_TYPE', 'P_SEX', 'P_USER', 'P_PSN', 'P_SAFE']
    
    tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    tmp_path = tmp.name
    tmp.close()
    
    df[mice_cols].to_csv(tmp_path, index=False)
    ro.globalenv['csv_path'] = tmp_path
    ro.globalenv['m'] = n_iter
    
    ro.r('''
        library(mice)
        r_df <- read.csv(csv_path, na.strings = c("NA",""))
        for (col in names(r_df)) r_df[[col]] <- as.factor(r_df[[col]])
        suppressMessages({
            imp <- mice(r_df, method=rep("polyreg",ncol(r_df)),
                        m=1, maxit=m, seed=42, printFlag=FALSE)
        })
        filled <- complete(imp)
        write.csv(filled, csv_path, row.names=FALSE)
    ''')
    
    filled = pd.read_csv(tmp_path)
    os.unlink(tmp_path)
    
    for col in mice_cols:
        df[col] = filled[col].values.astype(int)
    
    print("MICE imputation done")
    return df

In [ ]:
def simple_imputation(df):
    for col in ['P_AGE', 'C_HOUR']:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    
    if df['V_YEAR'].isnull().sum() > 0:
        v = df['V_YEAR'].mode(dropna=True)
        df['V_YEAR'] = df['V_YEAR'].fillna(v.iloc[0] if len(v) > 0 else df['V_YEAR'].median())
    
    for col in ['C_MNTH', 'C_WDAY']:
        if df[col].isnull().sum() > 0:
            v = df[col].mode(dropna=True)
            df[col] = df[col].fillna(v.iloc[0] if len(v) > 0 else df[col].median())
    
    print("Simple imputation done")
    return df

In [ ]:
# Apply preprocessing to train set
print("=" * 60)
print("PREPROCESSING TRAIN SET")
print("=" * 60)

df_train_prep = df_train.copy()
df_train_prep = story_imputation(df_train_prep)
df_train_prep = mice_imputation(df_train_prep, n_iter=5)
df_train_prep = simple_imputation(df_train_prep)

# Apply preprocessing to test set
print("\n" + "=" * 60)
print("PREPROCESSING TEST SET")
print("=" * 60)

df_test_prep = df_test.copy()
df_test_prep = story_imputation(df_test_prep)
df_test_prep = mice_imputation(df_test_prep, n_iter=5)
df_test_prep = simple_imputation(df_test_prep)

print(f"\nTrain set: {len(df_train_prep):,} samples")
print(f"Test set: {len(df_test_prep):,} samples")

## 6. Grid Search for Mixed Sampling

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, matthews_corrcoef
from imblearn.under_sampling import RandomUnderSampler
import tempfile
import rpy2.robjects as ro

In [ ]:
# Feature configuration
FEATURE_COLS = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_VEHS', 'V_YEAR', 'P_AGE']
NOMINAL_COLS = ['C_CONF', 'C_RCFG', 'C_RALN', 'C_TRAF', 'C_WTHR', 'C_RSUR', 'P_SAFE',
                'P_SEX', 'P_PSN', 'P_USER', 'V_TYPE']
TARGET = 'Fatality'

def prepare_features(df, ohe=None):
    df_model = df[FEATURE_COLS + NOMINAL_COLS + [TARGET]].copy()
    df_model = df_model.dropna().reset_index(drop=True)
    
    X_dense = csr_matrix(df_model[FEATURE_COLS].values)
    
    if ohe is None:
        ohe = OneHotEncoder(sparse_output=True, min_frequency=0.001, handle_unknown='infrequent_if_exist')
        X_ohe_sparse = ohe.fit_transform(df_model[NOMINAL_COLS])
    else:
        X_ohe_sparse = ohe.transform(df_model[NOMINAL_COLS])
    
    X_features = hstack([X_dense, X_ohe_sparse], format='csr')
    y = df_model[TARGET].astype(int).values
    
    return X_features, y, ohe

In [ ]:
def rose_r(X, y, sampling_strategy, random_state, nominal_indices=None):
    n_maj = int(np.sum(y == 0))
    n_min = max(1, int(np.sum(y == 1)))
    
    if sampling_strategy < 1:
        n_synth = max(0, int(n_maj * sampling_strategy) - n_min)
    else:
        n_synth = max(0, int(n_min * sampling_strategy) - n_min)
    
    if n_synth <= 0:
        return X, y
    
    n_total = n_maj + n_min + n_synth
    
    tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    tmp_path = tmp.name
    tmp.close()
    
    Xd = X.toarray() if hasattr(X, 'toarray') else X
    pd.DataFrame(np.column_stack([Xd, y.astype(int)])).to_csv(tmp_path, index=False, header=False)
    
    ro.globalenv['csv_path'] = tmp_path
    ro.globalenv['n_total'] = int(n_total)
    ro.globalenv['seed_val'] = int(random_state)
    
    if nominal_indices is not None:
        nom_idx_r = ro.IntVector([i + 1 for i in nominal_indices])
        ro.globalenv['nominal_indices'] = nom_idx_r
        ro.r('''
            library(ROSE)
            d <- read.csv(csv_path, header=FALSE)
            colnames(d)[ncol(d)] <- "y"
            d$y <- as.factor(d$y)
            for (i in nominal_indices) {
                d[,i] <- as.factor(d[,i])
            }
            r <- ROSE(y~., d, N=n_total, seed=seed_val)
            write.csv(r$data, csv_path, row.names=FALSE)
        ''')
    else:
        ro.r('''
            library(ROSE)
            d <- read.csv(csv_path, header=FALSE)
            colnames(d)[ncol(d)] <- "y"
            d$y <- as.factor(d$y)
            r <- ROSE(y~., d, N=n_total, seed=seed_val)
            write.csv(r$data, csv_path, row.names=FALSE)
        ''')
    
    rd = pd.read_csv(tmp_path)
    os.unlink(tmp_path)
    
    return rd.drop(columns=['y']).values.astype(float), rd['y'].values.astype(int)

In [ ]:
# Grid search parameters
UNDER_STRATEGIES = [0.3, 0.4, 0.5, 0.6]
ROSE_STRATEGIES = [0.5, 0.6, 0.7, 0.8, 0.9]

print("=" * 60)
print("GRID SEARCH FOR MIXED SAMPLING")
print("=" * 60)
print(f"Under strategies: {UNDER_STRATEGIES}")
print(f"ROSE strategies: {ROSE_STRATEGIES}")
print(f"Total combinations: {len(UNDER_STRATEGIES) * len(ROSE_STRATEGIES)}")

In [ ]:
# Prepare features for grid search
X_train, y_train, ohe = prepare_features(df_train_prep)
X_test, y_test, _ = prepare_features(df_test_prep, ohe=ohe)

# Calculate nominal indices
nominal_indices = list(range(len(FEATURE_COLS), len(FEATURE_COLS) + len(NOMINAL_COLS)))

print(f"Train features: {X_train.shape[1]}  Samples: {len(y_train):,}")
print(f"Test features: {X_test.shape[1]}  Samples: {len(y_test):,}")

In [ ]:
# Grid search
results = []

for under_strat in UNDER_STRATEGIES:
    for rose_strat in ROSE_STRATEGIES:
        print(f"\n>>> Under={under_strat}, ROSE={rose_strat}")
        
        try:
            # Apply mixed sampling
            Xm, ym = RandomUnderSampler(random_state=42, sampling_strategy=under_strat).fit_resample(X_train, y_train)
            X_sampled, y_sampled = rose_r(Xm, ym, rose_strat, 42, nominal_indices)
            
            print(f"  Sampled: {len(y_sampled):,} (fatal={y_sampled.sum():,}, rate={y_sampled.mean()*100:.3f}%)")
            
            # Quick evaluation
            model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
            model.fit(X_sampled, y_sampled)
            y_pred = model.predict(X_test)
            
            acc = accuracy_score(y_test, y_pred)
            tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            g_mean = np.sqrt(sens * spec) if (sens * spec) >= 0 else 0
            
            print(f"  Acc={acc:.3f} | Sens={sens:.3f} | Spec={spec:.3f} | F1={f1:.3f} | MCC={mcc:.3f} | G={g_mean:.3f}")
            
            results.append({
                'under_strategy': under_strat,
                'rose_strategy': rose_strat,
                'samples': len(y_sampled),
                'fatal_rate': y_sampled.mean(),
                'accuracy': acc,
                'sensitivity': sens,
                'specificity': spec,
                'f1': f1,
                'mcc': mcc,
                'g_mean': g_mean
            })
            
        except Exception as e:
            print(f"  ERROR: {e}")
            continue

In [ ]:
# Save and display results
df_results = pd.DataFrame(results)
grid_search_path = os.path.join(OUT_DIR, 'grid_search_results.csv')
df_results.to_csv(grid_search_path, index=False)
print(f"\nGrid search results saved to: {grid_search_path}")

print("\n" + "=" * 60)
print("TOP 5 BY MCC")
print("=" * 60)
print(df_results.nlargest(5, 'mcc')[['under_strategy', 'rose_strategy', 'mcc', 'accuracy', 'sensitivity', 'specificity', 'f1', 'g_mean']])

# Select best parameters (by MCC)
best_params = df_results.nlargest(1, 'mcc').iloc[0]
BEST_UNDER = best_params['under_strategy']
BEST_ROSE = best_params['rose_strategy']
print(f"\nBest parameters: under={BEST_UNDER}, rose={BEST_ROSE} (MCC={best_params['mcc']:.3f})")

## 7. Sampling Pipeline

In [ ]:
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek

SAMPLER_NAMES = ['no', 'under', 'rose', 'mixed', 'smote', 'borderline', 'smote_tomek', 'adasyn', 'nearmiss']

In [ ]:
def apply_sampler(X, y, name, seed=42, nominal_indices=None, under_strategy=None, rose_strategy=None):
    if name == 'no':
        return X, y
    if name == 'rose':
        strategy = rose_strategy if rose_strategy is not None else 0.5
        return rose_r(X, y, strategy, seed, nominal_indices)
    if name == 'mixed':
        under_strat = under_strategy if under_strategy is not None else 0.4
        rose_strat = rose_strategy if rose_strategy is not None else 0.8
        Xm, ym = RandomUnderSampler(random_state=seed, sampling_strategy=under_strat).fit_resample(X, y)
        return rose_r(Xm, ym, rose_strat, seed, nominal_indices)
    
    n_min = max(1, int(np.sum(y == 1)))
    n_maj = int(np.sum(y == 0))
    
    if n_min < 2 and name != 'under':
        print(f"  Skipped {name}: minority class too small")
        return X, y
    if n_maj < 2:
        print(f"  Skipped {name}: majority class too small")
        return X, y
    
    k = max(1, min(5, n_min - 1))
    
    if name == 'under':
        return RandomUnderSampler(random_state=seed).fit_resample(X, y)
    if name == 'smote':
        return SMOTE(random_state=seed, k_neighbors=k).fit_resample(X, y)
    if name == 'borderline':
        return BorderlineSMOTE(random_state=seed, k_neighbors=k).fit_resample(X, y)
    if name == 'smote_tomek':
        return SMOTETomek(random_state=seed, smote=SMOTE(k_neighbors=k)).fit_resample(X, y)
    if name == 'adasyn':
        return ADASYN(random_state=seed, n_neighbors=k).fit_resample(X, y)
    if name == 'nearmiss':
        return NearMiss(version=1).fit_resample(X, y)
    
    return X, y

In [ ]:
# Apply sampling techniques
print("=" * 60)
print("APPLYING SAMPLING TECHNIQUES")
print("=" * 60)

sampled_datasets = {}

for sampler_name in SAMPLER_NAMES:
    print(f"\n>>> {sampler_name.upper()} SAMPLING ...")
    
    X_sampled, y_sampled = apply_sampler(
        X_train, y_train, sampler_name, 42, nominal_indices,
        under_strategy=BEST_UNDER, rose_strategy=BEST_ROSE
    )
    
    print(f"  Samples: {len(y_sampled):,} (fatal={y_sampled.sum():,}, rate={y_sampled.mean()*100:.3f}%)")
    
    sampled_datasets[sampler_name] = (X_sampled, y_sampled)

## 8. Visualization

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import wasserstein_distance

In [ ]:
def compute_metrics(X, y):
    metrics = {}
    n_samples = len(y)
    
    if n_samples > 2 and len(np.unique(y)) > 1:
        try:
            metrics['silhouette'] = silhouette_score(X, y)
        except:
            metrics['silhouette'] = np.nan
        try:
            metrics['davies_bouldin'] = davies_bouldin_score(X, y)
        except:
            metrics['davies_bouldin'] = np.nan
        try:
            metrics['calinski_harabasz'] = calinski_harabasz_score(X, y)
        except:
            metrics['calinski_harabasz'] = np.nan
    else:
        metrics['silhouette'] = np.nan
        metrics['davies_bouldin'] = np.nan
        metrics['calinski_harabasz'] = np.nan
    
    X_0 = X[y == 0]
    X_1 = X[y == 1]
    if len(X_0) > 0 and len(X_1) > 0:
        try:
            wass_dists = []
            for i in range(X.shape[1]):
                wass = wasserstein_distance(X_0[:, i], X_1[:, i])
                wass_dists.append(wass)
            metrics['wasserstein'] = np.mean(wass_dists)
        except:
            metrics['wasserstein'] = np.nan
    else:
        metrics['wasserstein'] = np.nan
    
    return metrics

In [ ]:
# Scale data for visualization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.toarray())

# Fit PCA and UMAP
pca = PCA(n_components=2, random_state=42)
pca.fit(X_train_scaled)

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
reducer.fit_transform(X_train_scaled)

print("PCA and UMAP fitted on training data")

In [ ]:
# Visualize and compute metrics for each sampling technique
viz_dir = os.path.join(OUT_DIR, "visualizations")
os.makedirs(viz_dir, exist_ok=True)

metrics_results = []

for sampler_name, (X_s, y_s) in sampled_datasets.items():
    print(f"\n>>> {sampler_name.upper()} VISUALIZATION ...")
    
    # Scale sampled data
    X_s_dense = X_s.toarray() if hasattr(X_s, 'toarray') else X_s
    X_s_scaled = scaler.transform(X_s_dense)
    
    # PCA
    X_pca = pca.transform(X_s_scaled)
    
    # UMAP
    X_umap = reducer.transform(X_s_scaled)
    
    # Plot PCA
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = {0: '#1f77b4', 1: '#d62728'}
    labels = {0: 'Non-Fatality', 1: 'Fatality'}
    
    for cls in [0, 1]:
        mask = y_s == cls
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors[cls], label=f"{labels[cls]} (n={mask.sum():,})",
                   alpha=0.4, s=3, edgecolors='none')
    
    ax.set_title(f'PCA - {sampler_name.upper()} Sampling', fontsize=14)
    ax.legend(markerscale=5, fontsize=10)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    plt.tight_layout()
    plt.savefig(os.path.join(viz_dir, f"pca_{sampler_name}.png"), dpi=200)
    plt.close()
    
    # Plot UMAP
    fig, ax = plt.subplots(figsize=(10, 8))
    for cls in [0, 1]:
        mask = y_s == cls
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   c=colors[cls], label=f"{labels[cls]} (n={mask.sum():,})",
                   alpha=0.4, s=3, edgecolors='none')
    
    ax.set_title(f'UMAP - {sampler_name.upper()} Sampling', fontsize=14)
    ax.legend(markerscale=5, fontsize=10)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    plt.tight_layout()
    plt.savefig(os.path.join(viz_dir, f"umap_{sampler_name}.png"), dpi=200)
    plt.close()
    
    # Compute metrics
    metrics = compute_metrics(X_s_scaled, y_s)
    metrics['sampler'] = sampler_name
    metrics['n_samples'] = len(y_s)
    metrics['n_fatal'] = y_s.sum()
    metrics['fatality_rate'] = y_s.mean()
    metrics_results.append(metrics)
    
    print(f"  Metrics: Silhouette={metrics.get('silhouette', np.nan):.4f}  "
          f"DB={metrics.get('davies_bouldin', np.nan):.4f}  "
          f"CH={metrics.get('calinski_harabasz', np.nan):.4f}  "
          f"Wasserstein={metrics.get('wasserstein', np.nan):.4f}")

# Save metrics
df_metrics = pd.DataFrame(metrics_results)
metrics_path = os.path.join(OUT_DIR, "sampling_metrics.csv")
df_metrics.to_csv(metrics_path, index=False)
print(f"\nMetrics saved to: {metrics_path}")

## 9. Model Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score, average_precision_score

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    prauc = average_precision_score(y_test, y_prob)
    g_mean = np.sqrt(sens * spec) if (sens * spec) >= 0 else 0
    
    print(f"  {name:30s} | Acc={acc:.3f} | Sens={sens:.3f} | Spec={spec:.3f} | Prec={prec:.3f} | F1={f1:.3f} | MCC={mcc:.3f} | G={g_mean:.3f} | AUC={auc:.3f} | PR={prauc:.3f}")
    
    return {
        'Model': name,
        'Accuracy': acc,
        'Sensitivity': sens,
        'Specificity': spec,
        'Precision': prec,
        'F1': f1,
        'MCC': mcc,
        'G_mean': g_mean,
        'AUC_ROC': auc,
        'PR_AUC': prauc
    }

In [ ]:
# Evaluate all sampling techniques
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

all_results = []

for sampler_name, (X_train_s, y_train_s) in sampled_datasets.items():
    print(f"\n>>> {sampler_name.upper()} SAMPLING")
    
    # Lasso Regression
    print(f"  >>> Lasso Regression (5-fold CV) <<<")
    lasso = LogisticRegressionCV(
        penalty='l1', 
        solver='saga', 
        Cs=10, 
        cv=5, 
        max_iter=1000, 
        random_state=42, 
        n_jobs=-1
    )
    lasso.fit(X_train_s, y_train_s)
    result_lasso = evaluate_model(f'Lasso + {sampler_name}', lasso, X_test, y_test)
    all_results.append(result_lasso)
    
    # XGBoost
    print(f"  >>> XGBoost <<<")
    X_train_s_dense = X_train_s.toarray() if hasattr(X_train_s, 'toarray') else X_train_s
    X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else X_test
    
    spw = 1 if sampler_name == 'no' else max(1, sum(y_train_s == 0) / max(sum(y_train_s == 1), 1))
    xgb = XGBClassifier(
        n_estimators=1200, 
        learning_rate=0.5, 
        max_depth=3, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        scale_pos_weight=spw, 
        random_state=42, 
        use_label_encoder=False, 
        eval_metric='logloss', 
        n_jobs=-1
    )
    xgb.fit(X_train_s_dense, y_train_s)
    result_xgb = evaluate_model(f'XGBoost + {sampler_name}', xgb, X_test_dense, y_test)
    all_results.append(result_xgb)

In [ ]:
# Save results
df_results = pd.DataFrame(all_results)
results_path = os.path.join(OUT_DIR, 'model_evaluation_results.csv')
df_results.to_csv(results_path, index=False)
print(f"\nResults saved to: {results_path}")

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(df_results.to_string(index=False))

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print("Pipeline completed successfully!")
print("")
print("Key outputs:")
print(f"- Grid search results: {grid_search_path}")
print(f"- Sampling metrics: {metrics_path}")
print(f"- Model evaluation: {results_path}")
print(f"- Visualizations: {viz_dir}/")
print("")
print("Best mixed sampling parameters:")
print(f"- under_strategy: {BEST_UNDER}")
print(f"- rose_strategy: {BEST_ROSE}")